# Nuclear-to-Cell Expansion with `celldega.nbhd`

This notebook is a runnable companion to a nucleus/cell segmentation-sensitivity
analysis: starting from a nucleus polygon, grow it outward in fixed steps until it
reaches the boundary of its corresponding (larger) cell segmentation, and compute a
cell-by-gene matrix at every step -- the original nucleus radius plus each expanded
radius.

That workflow is now a first-class part of Celldega's neighborhood API:

- **`NeighborhoodCollection.calc_expansion`** replaces the manual
  `expand_nuclei_within_cell` buffering loop. It is deliberately generic: give it a
  `NeighborhoodCollection` of *any* entity and a matching per-entity bounding
  `GeoDataFrame`; it buffers every entity outward at each requested radius (in
  microns), clips each one to its own bound so growth never overshoots it, and
  returns one new `NeighborhoodCollection` per radius, all sharing the same
  observation axis so results stay directly comparable across radii. Nucleus ->
  cell is just the running example below -- the same method works for any other
  pair of nested per-entity geometries.
- **`NeighborhoodCollection.calc_signature(by="cell-free", ...)`** replaces the
  custom `assign_trx_to_entity_streaming_parquet_optimized` + manual pivot. It
  spatially joins transcripts to each radius's polygons and returns a cell-by-gene
  `AnnData`, ready for the usual scanpy pipeline -- either from an in-memory
  `GeoDataFrame` (`gdf_trx=`) or streamed straight from a parquet file
  (`trx_parquet_path=`) for transcript files too large to hold in memory.
- **`celldega.nbhd.gdf_from_contour_coords`** builds a nucleus/cell
  `GeoDataFrame` directly from a long-format vertex-coordinate table (the shape
  of a `*_contour_coords.csv` export), replacing a hand-rolled `safe_polygon` +
  `groupby(...).agg(list)` helper.
- **`celldega.nbhd.df_to_anndata`** wraps any entity-by-gene (or other matrix)
  DataFrame as a bare `AnnData`, with no normalization/PCA/clustering computed.

Because the real instrument files (OME-TIFF, per-dataset contour CSVs, a full-tile
`transcripts.parquet`) aren't available here, this notebook builds a small
**synthetic** nucleus/cell/transcript dataset with the same shape as a real
segmentation export, so every cell below runs standalone. The final section maps
each synthetic variable back to the real pipeline's inputs so you can swap in your
own paths.

In [1]:
import os
import tempfile

import numpy as np
import pandas as pd
import geopandas as gpd
import scanpy as sc
import matplotlib.pyplot as plt
from shapely.geometry import Point

import celldega as dega

dega.__version__

'0.18.0'

## 1. Nucleus + cell-boundary polygons

A real pipeline builds these from segmentation contour CSVs (one polygon per cell,
in each of a nucleus file and an expanded-cell-boundary file). Here we synthesize
the same shape: two `GeoDataFrame`s sharing a `cell_id` column, one with a small
nucleus polygon per cell and one with its larger enclosing cell polygon.

In [2]:
rng = np.random.default_rng(0)

N_ROWS, N_COLS = 10, 12
SPACING_UM = 20.0

records_nuclei, records_cells, cell_meta = [], [], []

cell_id = 0
for row in range(N_ROWS):
    for col in range(N_COLS):
        cx = col * SPACING_UM + rng.normal(0, 1.5)
        cy = row * SPACING_UM + rng.normal(0, 1.5)

        cell_radius = rng.uniform(7.0, 9.0)
        nucleus_radius = rng.uniform(2.5, 3.5)
        jitter = rng.uniform(0, 2.0, size=2)
        nx, ny = cx + jitter[0], cy + jitter[1]

        # two synthetic "cell types" so downstream clustering has real structure
        cell_type = "TypeA" if (row + col) % 2 == 0 else "TypeB"

        records_nuclei.append(
            {"cell_id": cell_id, "geometry": Point(nx, ny).buffer(nucleus_radius, resolution=12)}
        )
        records_cells.append(
            {"cell_id": cell_id, "geometry": Point(cx, cy).buffer(cell_radius, resolution=12)}
        )
        cell_meta.append(
            {"cell_id": cell_id, "cell_type": cell_type, "cx": cx, "cy": cy,
             "nx": nx, "ny": ny, "nucleus_radius": nucleus_radius, "cell_radius": cell_radius}
        )
        cell_id += 1

gdf_nuclei = gpd.GeoDataFrame(records_nuclei)
gdf_cells = gpd.GeoDataFrame(records_cells)
df_cell_meta = pd.DataFrame(cell_meta).set_index("cell_id")

gdf_nuclei.shape, gdf_cells.shape

((120, 2), (120, 2))

### Building from segmentation contour CSVs

The synthetic `gdf_nuclei`/`gdf_cells` above were built directly with Shapely.
A real pipeline more often exports segmentation contours to CSV instead -- one
row per polygon vertex, grouped by a cell id (e.g. a `*_nuclei_contour_coords.csv`
/ `*_cell_contour_coords.csv` pair). `celldega.nbhd.gdf_from_contour_coords`
builds a `GeoDataFrame` directly from that long format, so you don't need to
hand-roll a `safe_polygon` + `groupby(...).agg(list)` helper yourself. Any
coordinate offsetting/rescaling (e.g. registering instrument microns into an
image's pixel space) should happen on the vertex columns before calling it.

In [3]:
# a toy long-format contour table, matching a *_nuclei_contour_coords.csv shape --
# reuses one nucleus'''s own vertices, just to show the round trip
example_geom = gdf_nuclei.geometry.iloc[0]
vx, vy = zip(*example_geom.exterior.coords)
df_contours_demo = pd.DataFrame({
    "cell_id": [gdf_nuclei["cell_id"].iloc[0]] * len(vx),
    "vertex_x": vx,
    "vertex_y": vy,
})

gdf_from_csv_demo = dega.nbhd.gdf_from_contour_coords(df_contours_demo)
gdf_from_csv_demo

,cell_id,geometry,center_x,center_y
0,0,"POLYGON ((4.33166 1.62735, 4.31013 1.29888, 4....",1.815136,1.627354


## 2. Wrap the nuclei as a `NeighborhoodCollection`

Each nucleus becomes one observation ("neighborhood"), keyed by `cell_id`.

In [4]:
nbhd_nuclei = dega.nbhd.NeighborhoodCollection(
    gdf=gdf_nuclei, nbhd_type="nucleus", nbhd_col="cell_id"
)
nbhd_nuclei.obs[["area_um2"]].head()

,area_um2
neighborhood_id,
0,19.838659
1,36.964149
2,22.426905
3,21.573983
4,37.955601


## 3. Expansion series

`calc_expansion` buffers every nucleus outward at each radius in
`radii_um` and intersects it with the matching row of `gdf_cells`, so a nucleus
never grows past its own cell's membrane. It returns a dict keyed by radius, each
value a new `NeighborhoodCollection` sharing the same `cell_id` observation axis.

In [5]:
radii_um = [0, 0.5, 1, 1.5, 2, 2.5, 3]
nbhd_series = nbhd_nuclei.calc_expansion(gdf_cells, radii_um=radii_um)

for radius, nbhd in nbhd_series.items():
    print(f"radius={radius:>4} um -> n={len(nbhd.gdf):>4}  mean_area={nbhd.gdf['area_um2'].mean():6.2f} um^2")

radius= 0.0 um -> n= 120  mean_area= 28.34 um^2
radius= 0.5 um -> n= 120  mean_area= 38.52 um^2
radius= 1.0 um -> n= 120  mean_area= 50.28 um^2
radius= 1.5 um -> n= 120  mean_area= 63.58 um^2
radius= 2.0 um -> n= 120  mean_area= 78.40 um^2
radius= 2.5 um -> n= 120  mean_area= 94.52 um^2
radius= 3.0 um -> n= 120  mean_area=111.39 um^2


In [6]:
# visual sanity check for one example cell, mirroring the original notebook's plot
example_id = str(int(df_cell_meta.index[7]))

fig, axes = plt.subplots(1, len(radii_um), figsize=(3 * len(radii_um), 3))
for ax, radius in zip(axes, radii_um):
    nbhd = nbhd_series[radius]
    gdf_cells[gdf_cells["cell_id"].astype(str) == example_id].boundary.plot(ax=ax, color="black")
    nbhd.gdf.loc[[example_id]].plot(ax=ax, color="lightblue", edgecolor="blue", alpha=0.7)
    ax.set_title(f"+{radius} um")
    ax.set_aspect("equal")
    ax.axis("off")
fig.suptitle(f"cell_id {example_id}")
fig.tight_layout()

### Working in pixel space

Segmentation pipelines often store nucleus/cell polygons in image-pixel
coordinates rather than microns -- e.g. the original notebook builds them via
`vertex_x * high_res_scale`, where `high_res_scale = 1 / scaling_factor` is a
pixels-per-micron factor (`scaling_factor` itself, from `PhysicalSizeX`, is
microns-per-pixel). `calc_expansion` needs to know that scale to convert
`radii_um` into the geometry's own units before buffering.

Pass whichever factor your pipeline already has on hand:

- `scale_um_per_pixel=` for a microns-per-pixel factor (e.g. OME-XML `PhysicalSizeX`) -- a micron distance is *divided* by this to get pixels.
- `pixels_per_micron=` for the reciprocal, pixels-per-micron convention (e.g. `high_res_scale` above) -- a micron distance is *multiplied* by this to get pixels, matching `buffer_dist = expand_um * high_res_scale` directly.

Both are shown below on the same nuclei, scaled up into a toy "pixel" coordinate
space, and confirmed to reproduce the same real-world (micron) areas as the
micron-space series computed earlier.

In [7]:
high_res_scale = 4.0  # pixels per micron, e.g. derived from an OME-XML PhysicalSizeX

# stand-in for "already-in-pixel-space" geometry, as produced by a real
# segmentation pipeline's `vertex_x * high_res_scale` step
gdf_nuclei_px = gdf_nuclei.copy()
gdf_nuclei_px["geometry"] = gdf_nuclei_px.geometry.scale(high_res_scale, high_res_scale, origin=(0, 0))
gdf_cells_px = gdf_cells.copy()
gdf_cells_px["geometry"] = gdf_cells_px.geometry.scale(high_res_scale, high_res_scale, origin=(0, 0))

nbhd_nuclei_px = dega.nbhd.NeighborhoodCollection(
    gdf=gdf_nuclei_px, nbhd_type="nucleus", nbhd_col="cell_id"
)
nbhd_series_px = nbhd_nuclei_px.calc_expansion(
    gdf_cells_px,
    radii_um=radii_um,
    is_pixel_space=True,
    pixels_per_micron=high_res_scale,  # same variable your own notebook already computes
)

micron_areas = pd.Series({r: nbhd.gdf["area_um2"].sum() for r, nbhd in nbhd_series.items()}).sort_index()
pixel_areas = pd.Series({r: nbhd.gdf["area_um2"].sum() for r, nbhd in nbhd_series_px.items()}).sort_index()
pd.testing.assert_series_equal(micron_areas, pixel_areas, check_names=False, rtol=1e-6)
print("pixel-space geometry + pixels_per_micron reproduces the micron-space result")

pixel-space geometry + pixels_per_micron reproduces the micron-space result


## 4. Synthetic transcripts

Stand-in for a `transcripts.parquet` with non-Xenium columns -- `x`, `y`, `name` --
matching the columns used in the original notebook's
`assign_trx_to_entity_streaming_parquet_optimized(..., x_col="x", y_col="y",
gene_col="name")` call. Two gene pairs simulate real biology: `NucGene*`
transcripts cluster tightly at the nucleus center (captured at every radius), while
`CytoGene*` and a cell-type marker gene (`MarkerA`/`MarkerB`) scatter through the
cytoplasm and are only picked up as the buffer radius grows.

We also write these to an actual `transcripts.parquet` file, so the streaming
API below reads from disk exactly like it would on a real, whole-tile transcript
file.

In [8]:
trx_rows = []
for cid, meta in df_cell_meta.iterrows():
    n_nuc = rng.poisson(15)
    nuc_xy = rng.normal([meta["nx"], meta["ny"]], meta["nucleus_radius"] / 3, size=(n_nuc, 2))
    nuc_genes = rng.choice(["NucGene1", "NucGene2"], size=n_nuc)

    # rejection-sample points in the cytoplasm annulus (inside cell, outside nucleus)
    cyto_xy = []
    while len(cyto_xy) < 20:
        theta = rng.uniform(0, 2 * np.pi)
        r = meta["cell_radius"] * np.sqrt(rng.uniform(0, 1))
        x, y = meta["cx"] + r * np.cos(theta), meta["cy"] + r * np.sin(theta)
        if (x - meta["nx"]) ** 2 + (y - meta["ny"]) ** 2 > meta["nucleus_radius"] ** 2:
            cyto_xy.append((x, y))
    cyto_xy = np.array(cyto_xy)
    cyto_genes = rng.choice(["CytoGene1", "CytoGene2"], size=len(cyto_xy))

    marker_gene = "MarkerA" if meta["cell_type"] == "TypeA" else "MarkerB"
    marker_xy = cyto_xy[rng.integers(0, len(cyto_xy), size=rng.poisson(10))]

    for xy, gene in zip(nuc_xy, nuc_genes):
        trx_rows.append({"x": xy[0], "y": xy[1], "name": gene})
    for xy, gene in zip(cyto_xy, cyto_genes):
        trx_rows.append({"x": xy[0], "y": xy[1], "name": gene})
    for xy in marker_xy:
        trx_rows.append({"x": xy[0], "y": xy[1], "name": marker_gene})

df_trx = pd.DataFrame(trx_rows)
gdf_trx = gpd.GeoDataFrame(df_trx[["name"]], geometry=gpd.points_from_xy(df_trx["x"], df_trx["y"]))

# persist to a real parquet file, so the streaming API (below) reads from disk
# the same way it would for a real, whole-tile transcripts.parquet
trx_parquet_path = os.path.join(tempfile.mkdtemp(), "transcripts.parquet")
df_trx.to_parquet(trx_parquet_path)

gdf_trx.shape, gdf_trx["name"].value_counts().to_dict()

((5490, 2),
 {np.str_('CytoGene1'): 1243,
  np.str_('CytoGene2'): 1157,
  np.str_('NucGene1'): 984,
  np.str_('NucGene2'): 837,
  'MarkerA': 644,
  'MarkerB': 625})

## 5. Cell-by-gene matrix at every radius

`calc_signature(by="cell-free", ...)` spatially joins transcripts to each
radius's polygons and returns transcript counts as an `AnnData` in
`nbhd.mod["gene_cell_free"]` -- one call per radius, no custom pivot code
needed. It supports the same transcript source in two ways:

- **In-memory** (`gdf_trx=...`): loads the whole transcript table into memory as
  points and does a single spatial join. Simple, and fine when the transcripts
  already fit comfortably in memory (or you've pre-filtered them yourself).
- **Streaming** (`trx_parquet_path=...`): reads the parquet file in batches via
  `pyarrow`, narrowing candidate entities per batch with a spatial index before
  testing exact polygons -- the same mechanics as the original notebook's
  `assign_trx_to_entity_streaming_parquet_optimized`. Use this for a real,
  whole-tile `transcripts.parquet` (tens of millions of rows) that you don't want
  to load into memory seven times over (once per radius).

Both produce identical counts -- the cell below runs the streaming path (as you
would on real data) and spot-checks it against the in-memory path for one radius.

In [9]:
for radius, nbhd in nbhd_series.items():
    nbhd.calc_signature(
        by="cell-free",
        trx_parquet_path=trx_parquet_path,
        x_col="x",
        y_col="y",
        feature_col="name",
        drop_missing=False,
    )

gene_totals = pd.DataFrame(
    {
        radius: pd.DataFrame(
            nbhd.mod["gene_cell_free"].X, columns=nbhd.mod["gene_cell_free"].var_names
        ).sum()
        for radius, nbhd in nbhd_series.items()
    }
).T
gene_totals

Calculating neighborhood-by-gene (cell-free, streaming parquet)
Calculating neighborhood-by-gene (cell-free, streaming parquet)


Calculating neighborhood-by-gene (cell-free, streaming parquet)
Calculating neighborhood-by-gene (cell-free, streaming parquet)
Calculating neighborhood-by-gene (cell-free, streaming parquet)
Calculating neighborhood-by-gene (cell-free, streaming parquet)
Calculating neighborhood-by-gene (cell-free, streaming parquet)


,CytoGene1,CytoGene2,MarkerA,MarkerB,NucGene1,NucGene2
0.0,NaN,NaN,NaN,NaN,970.0,824.0
0.5,82.0,79.0,37.0,41.0,982.0,834.0
1.0,171.0,162.0,76.0,75.0,984.0,837.0
1.5,274.0,255.0,118.0,121.0,984.0,837.0
2.0,379.0,367.0,170.0,172.0,984.0,837.0
2.5,485.0,463.0,228.0,217.0,984.0,837.0
3.0,610.0,571.0,291.0,284.0,984.0,837.0


### A bare AnnData, without Celldega's cat/color bookkeeping

`calc_signature` already returns a ready-to-use `AnnData` in
`nbhd.mod["gene_cell_free"]` (with `n_transcripts`, `cat`, `color` metadata
attached). If you already have your own entity-by-gene DataFrame -- built by
hand, or from a lower-level function directly -- `celldega.nbhd.df_to_anndata`
wraps it as a plain `AnnData` (`obs` = the DataFrame's index, `var` = its
columns, `X` = its values), with no normalization, PCA, neighbors, or
clustering computed.

In [10]:
df_gene_counts = pd.DataFrame(
    nbhd_series[3.0].mod["gene_cell_free"].X,
    index=nbhd_series[3.0].mod["gene_cell_free"].obs_names,
    columns=nbhd_series[3.0].mod["gene_cell_free"].var_names,
)
adata_bare = dega.nbhd.df_to_anndata(df_gene_counts)
adata_bare

AnnData object with n_obs × n_vars = 120 × 6

### Streaming vs. in-memory sanity check

Confirms the streaming path above (`trx_parquet_path=`) and the in-memory path
(`gdf_trx=`) agree, using the radius=3um collection as a spot check.

In [11]:
nbhd_check = nbhd_series[3.0]
nbhd_check.calc_signature(
    by="cell-free", gdf_trx=gdf_trx, feature_col="name",
    modality_name="gene_cell_free_in_memory", drop_missing=False,
)

streamed = pd.DataFrame(
    nbhd_check.mod["gene_cell_free"].X, columns=nbhd_check.mod["gene_cell_free"].var_names,
    index=nbhd_check.mod["gene_cell_free"].obs_names,
)
in_memory = pd.DataFrame(
    nbhd_check.mod["gene_cell_free_in_memory"].X,
    columns=nbhd_check.mod["gene_cell_free_in_memory"].var_names,
    index=nbhd_check.mod["gene_cell_free_in_memory"].obs_names,
)
assert streamed.equals(in_memory[streamed.columns])
print("streaming and in-memory paths agree")

Calculating neighborhood-by-gene (cell-free, provided gdf_trx)
streaming and in-memory paths agree


/Users/jishar/Documents/celldega/dega/lib/python3.12/site-packages/mudata/_core/mudata.py:931: UserWarning: Cannot join columns with the same name because var_names are intersecting.
  warnings.warn(


### Transcript totals via `calc_transcript_assignment`

For just a per-entity transcript *total* (no gene breakdown), the same streaming
join backs `calc_transcript_assignment(trx_parquet_path=...)` -- e.g. as a quick
QC pass before committing to a full `calc_signature` call at every radius.

In [12]:
nbhd_series[3.0].calc_transcript_assignment(
    trx_parquet_path=trx_parquet_path, x_col="x", y_col="y", gene_col="name"
)
nbhd_series[3.0].obs[["total_transcripts"]].head()

,total_transcripts
neighborhood_id,
0,33
1,29
2,23
3,37
4,34


## 6. Downstream analysis per radius

The same scanpy pipeline as the original notebook (normalize, log1p, scale, PCA,
neighbors, Leiden, UMAP), run once per radius on `nbhd.mod["gene_cell_free"]`.
Pipeline parameters (`n_comps`, `n_neighbors`) are scaled down here for this small
synthetic demo -- use your usual settings (e.g. `n_top_genes=5000`,
`n_neighbors=30`) on real data.

In [13]:
adatas = {}
for radius, nbhd in nbhd_series.items():
    adata = nbhd.mod["gene_cell_free"].copy()
    adata.X = adata.X.astype("float32")
    adata.obs["cell_type"] = df_cell_meta.loc[adata.obs_names.astype(int), "cell_type"].to_numpy()

    sc.pp.normalize_total(adata)
    sc.pp.log1p(adata)
    sc.pp.scale(adata, max_value=10)

    n_comps = min(5, adata.n_vars - 1, adata.n_obs - 1)
    sc.tl.pca(adata, n_comps=n_comps, random_state=0)
    sc.pp.neighbors(adata, n_neighbors=10, use_rep="X_pca", random_state=0)
    sc.tl.leiden(adata, flavor="igraph", key_added="leiden", resolution=0.5, random_state=0)
    sc.tl.umap(adata, random_state=0)

    adatas[radius] = adata

sc.pl.umap(adatas[3.0], color=["leiden", "cell_type"], title=[f"radius=3um: leiden", f"radius=3um: true cell_type"])

/Users/jishar/Documents/celldega/dega/lib/python3.12/site-packages/scipy/sparse/_index.py:216: SparseEfficiencyWarning: Changing the sparsity structure of a csr_matrix is expensive. lil and dok are more efficient.
  self._set_arrayXarray(i, j, x)


/Users/jishar/Documents/celldega/dega/lib/python3.12/site-packages/scanpy/plotting/_utils.py:364: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 7. Compare across radii

Nuclear genes are already fully captured at radius 0; cytoplasmic and marker genes
climb steadily as the buffer reaches further into the cell. Clustering into the two
true cell types only stabilizes once enough cytoplasmic/marker signal is
captured.

In [14]:
n_clusters = {radius: adata.obs["leiden"].nunique() for radius, adata in adatas.items()}

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(gene_totals.index, gene_totals["NucGene1"] + gene_totals["NucGene2"], "o-", label="nuclear genes")
axes[0].plot(gene_totals.index, gene_totals["CytoGene1"] + gene_totals["CytoGene2"], "o-", label="cytoplasmic genes")
axes[0].plot(gene_totals.index, gene_totals["MarkerA"] + gene_totals["MarkerB"], "o-", label="marker genes")
axes[0].set_xlabel("buffer radius (um)")
axes[0].set_ylabel("total transcripts captured")
axes[0].legend()
axes[0].set_title("Transcript capture vs. nuclear buffer radius")

axes[1].plot(list(n_clusters.keys()), list(n_clusters.values()), "o-", color="crimson")
axes[1].set_xlabel("buffer radius (um)")
axes[1].set_ylabel("n leiden clusters")
axes[1].set_title("Cluster count vs. nuclear buffer radius")
fig.tight_layout()

## Mapping this onto a real pipeline

| Original notebook | This notebook / Celldega API |
| --- | --- |
| `safe_polygon` + `groupby("cell_id").agg(list)` parsing `..._nuclei_contour_coords.csv` | `dega.nbhd.gdf_from_contour_coords(df_contours, id_col="cell_id", x_col="vertex_x", y_col="vertex_y")` |
| `gdf_nuclei_original` (parsed from `..._nuclei_contour_coords.csv`) | `gdf_nuclei` -> `NeighborhoodCollection(gdf=gdf_nuclei, nbhd_col="cell_id")` |
| `gdf_cells` / `gdf_cells2` (parsed from `..._Expanded_5um_cell_contour_coords.csv`) | `gdf_cells` passed to `calc_expansion` |
| `expand_nuclei_within_cell(nuclei_gdf, expand_um)` loop building `nuclei_gdfs = {"original": ..., "expanded_0_5um": ..., ...}` | `nbhd_series = nbhd_nuclei.calc_expansion(gdf_cells, radii_um=[0, 0.5, 1, 1.5, 2, 2.5, 3])` |
| `assign_trx_to_entity_streaming_parquet_optimized(trx_parquet_path, entity_gdf, x_col="x", y_col="y", gene_col="name", batch_size=1_000_000)` + manual `pivot_table` per radius | `nbhd.calc_signature(by="cell-free", trx_parquet_path=trx_parquet_path, x_col="x", y_col="y", feature_col="name", batch_size=1_000_000)` per radius -- same batched-parquet-plus-spatial-index mechanics, now built in |
| `assignments_out=...` (per-transcript assignment parquet) | not written by `calc_signature` (it only needs the resulting counts); if you need per-transcript assignments too, keep using your own writer alongside it |
| Just the transcript *total* per entity, no gene breakdown | `nbhd.calc_transcript_assignment(trx_parquet_path=trx_parquet_path, x_col="x", y_col="y", gene_col="name")` -> adds a `total_transcripts` column to `nbhd.obs` via the same streaming join |
| Per-radius `pd.read_parquet(..._nuclei_by_gene.parquet)` -> `AnnData` (e.g. `ad.AnnData(X=nbg)`) | `nbhd.mod["gene_cell_free"]` (already an `AnnData`), or `dega.nbhd.df_to_anndata(your_own_df)` for a bare one with no Celldega bookkeeping |
| Per-radius `adata.write(...h5ad)` | `nbhd.mod["gene_cell_free"].write_h5ad(...)`, or persist the whole collection (geometry + all modalities) with `nbhd.write("radius.h5mu")` |

`calc_signature` also accepts `gdf_trx=` (an already-in-memory `GeoDataFrame` of
transcript points) for smaller or pre-filtered transcript sources -- see the
sanity-check cell above, which confirms both paths agree. Use `trx_parquet_path=`
whenever the transcripts don't comfortably fit in memory, especially since an
expansion series re-joins the same transcripts once per radius.